# 衛星バンド追加 - Sentinel-2 スペクトル指数

`train_composite/` 以下の Sentinel-2 TIF に、`docs/satellite_feature_plan.md` で定義した指数バンドを追加し、
`train_composite-add-features/` に保存します。

追加する指数（この順番で `descriptions` に追加）:
1. NDVI
2. EVI
3. NDBI
4. UI
5. NDWI
6. MNDWI
7. BSI
8. IBI
9. Clay (B11/B12)
10. IronOxide (B4/B2)
11. Carbonate (B11/B8A)
12. AerosolIndex (B1/B4)

In [ ]:
import os
import glob
import shutil
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

TRAIN_DIR   = '../data/train_dataset_82ddf14911a54c729380209510ae25ac/train_composite'
OUTPUT_DIR  = '../data/train_dataset_82ddf14911a54c729380209510ae25ac/train_composite-add-features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Input : {TRAIN_DIR}')
print(f'Output: {OUTPUT_DIR}')

## バンドインデックスマッピング

In [ ]:
# Sentinel-2 バンド名 -> rasterio バンド番号（1-indexed）
BAND_MAP = {
    'B1':  1,
    'B2':  2,
    'B3':  3,
    'B4':  4,
    'B5':  5,
    'B6':  6,
    'B7':  7,
    'B8':  8,
    'B8A': 9,
    'B9':  10,
    'B11': 11,
    'B12': 12,
}

# 追加する指数の定義（name, formula_fn）
# formula_fn: bands dict -> np.ndarray
EPS = 1e-10  # ゼロ除算防止

INDEX_DEFS = [
    ('NDVI',        lambda b: (b['B8'] - b['B4']) / (b['B8'] + b['B4'] + EPS)),
    ('EVI',         lambda b: 2.5 * (b['B8'] - b['B4']) / (b['B8'] + 6*b['B4'] - 7.5*b['B2'] + 1 + EPS)),
    ('NDBI',        lambda b: (b['B11'] - b['B8']) / (b['B11'] + b['B8'] + EPS)),
    ('UI',          lambda b: (b['B7'] - b['B4']) / (b['B7'] + b['B4'] + EPS)),
    ('NDWI',        lambda b: (b['B3'] - b['B8']) / (b['B3'] + b['B8'] + EPS)),
    ('MNDWI',       lambda b: (b['B3'] - b['B11']) / (b['B3'] + b['B11'] + EPS)),
    ('BSI',         lambda b: ((b['B11'] + b['B4']) - (b['B8'] + b['B2'])) /
                              ((b['B11'] + b['B4']) + (b['B8'] + b['B2']) + EPS)),
    ('IBI',         None),  # 後で NDVI, MNDWI, NDBI を使って計算
    ('Clay',        lambda b: b['B11'] / (b['B12'] + EPS)),
    ('IronOxide',   lambda b: b['B4'] / (b['B2'] + EPS)),
    ('Carbonate',   lambda b: b['B11'] / (b['B8A'] + EPS)),
    ('AerosolIndex',lambda b: b['B1'] / (b['B4'] + EPS)),
]

INDEX_NAMES = [name for name, _ in INDEX_DEFS]
print('追加指数:', INDEX_NAMES)

## 処理関数

In [ ]:
def compute_indices(bands: dict) -> list:
    """バンド辞書から全指数を計算してリストで返す"""
    results = []
    ndvi  = (bands['B8'] - bands['B4']) / (bands['B8'] + bands['B4'] + EPS)
    mndwi = (bands['B3'] - bands['B11']) / (bands['B3'] + bands['B11'] + EPS)
    ndbi  = (bands['B11'] - bands['B8']) / (bands['B11'] + bands['B8'] + EPS)

    for name, fn in INDEX_DEFS:
        if name == 'IBI':
            numer = ndbi - (ndvi + mndwi) / 2
            denom = ndbi + (ndvi + mndwi) / 2 + EPS
            arr = numer / denom
        else:
            arr = fn(bands)
        results.append(arr.astype(np.float64))
    return results


def process_sentinel(src_path: str, dst_path: str) -> str:
    """Sentinel-2 TIF を読み込み、指数バンドを追加して保存する"""
    with rasterio.open(src_path) as src:
        meta = src.meta.copy()
        orig_descs = list(src.descriptions)

        # 全バンドを辞書として読み込む
        bands = {}
        for band_name, band_idx in BAND_MAP.items():
            bands[band_name] = src.read(band_idx).astype(np.float64)

    # 指数を計算
    index_arrays = compute_indices(bands)

    # 出力メタデータを更新
    new_count = meta['count'] + len(INDEX_NAMES)
    meta.update(count=new_count, dtype='float64')

    # 保存
    with rasterio.open(dst_path, 'w', **meta) as dst:
        # 元の12バンドを書き込む
        with rasterio.open(src_path) as src:
            for i in range(1, 13):
                dst.write(src.read(i).astype(np.float64), i)

        # 指数バンドを追加
        for j, arr in enumerate(index_arrays):
            dst.write(arr, 13 + j)

        # descriptions を設定（元のバンド名 + 指数名）
        dst.descriptions = tuple(orig_descs + INDEX_NAMES)

    return dst_path


def process_viirs(src_path: str, dst_path: str) -> str:
    """VIIRS TIF はそのままコピーする（Sentinel-2 指数は適用不可）"""
    shutil.copy2(src_path, dst_path)
    return dst_path

## 動作確認（1ファイル）

In [ ]:
sample_src = sorted(glob.glob(f'{TRAIN_DIR}/sentinel_2_*.tif'))[0]
sample_dst = os.path.join(OUTPUT_DIR, os.path.basename(sample_src))

process_sentinel(sample_src, sample_dst)

with rasterio.open(sample_dst) as dst:
    print(f'Bands : {dst.count}')
    print(f'Descriptions: {dst.descriptions}')
    print(f'Shape : {dst.shape}')
    print(f'Dtype : {dst.dtypes[0]}')

## 全ファイル処理

In [ ]:
sentinel_files = sorted(glob.glob(f'{TRAIN_DIR}/sentinel_2_*.tif'))
viirs_files    = sorted(glob.glob(f'{TRAIN_DIR}/viirs_*.tif'))

print(f'Sentinel-2 files: {len(sentinel_files)}')
print(f'VIIRS files     : {len(viirs_files)}')

In [ ]:
# Sentinel-2: 指数バンドを追加して保存
errors = []
for src_path in tqdm(sentinel_files, desc='Sentinel-2'):
    fname = os.path.basename(src_path)
    dst_path = os.path.join(OUTPUT_DIR, fname)
    try:
        process_sentinel(src_path, dst_path)
    except Exception as e:
        errors.append((fname, str(e)))

print(f'Done. Errors: {len(errors)}')
for fname, err in errors:
    print(f'  {fname}: {err}')

In [ ]:
# VIIRS: そのままコピー
for src_path in tqdm(viirs_files, desc='VIIRS (copy)'):
    fname = os.path.basename(src_path)
    dst_path = os.path.join(OUTPUT_DIR, fname)
    process_viirs(src_path, dst_path)

print('VIIRS copy done.')

## 出力確認

In [ ]:
output_files = glob.glob(f'{OUTPUT_DIR}/*.tif')
print(f'出力ファイル数: {len(output_files)}')

# Sentinel サンプルの確認
sample = sorted(glob.glob(f'{OUTPUT_DIR}/sentinel_2_*.tif'))[0]
with rasterio.open(sample) as src:
    print(f'\n[Sentinel-2 サンプル] {os.path.basename(sample)}')
    print(f'  Bands       : {src.count}')
    print(f'  Descriptions: {src.descriptions}')

# VIIRS サンプルの確認
sample_v = sorted(glob.glob(f'{OUTPUT_DIR}/viirs_*.tif'))[0]
with rasterio.open(sample_v) as src:
    print(f'\n[VIIRS サンプル] {os.path.basename(sample_v)}')
    print(f'  Bands       : {src.count}')
    print(f'  Descriptions: {src.descriptions}')